# THE CURRENT SCORING IS BASED ON:

1. Signals 
2. Sector coefficient
3. Territorial coefficient
4. Chiffre d’affaires (CA) - but no weight was given to this one
5. Taux de croissance du CA - but no weight was given to this one
6. Age de l’entreprise

0 → if 0–3 years
+2 → if 3–10 years
+5 → if >20 years
(10–20 years not specified → we will assume +2 unless you prefer +0)

7. Nombre d’établissements

+5 → if multisites

8. Forme juridique

+5 → if SA or SAS (companies with stronger capacity to buy)

In [ ]:
def score_sector(sector):
    mapping = {
        'industrie_manufacturière': 1.0,
        'agroalimentaire': 1.2,
        'energie_chimie_pharma': 1.3,
        'logistique_transport': 0.9,
        'esn_services_conseil_logiciel': 0.6
    }
    return mapping.get(sector, 1.0)  # default = 1.0


In [ ]:
def score_territory(zone):
    mapping = {
        'zone_industrielle_dynamique': 1.1,
        'zone_rurale_neutre': 1.0,
        'zone_ en_decroissance': 0.8
    }
    return mapping.get(zone, 1.0)


In [ ]:
def score_age(age):
    if age <= 3:
        return 0
    elif age <= 10:
        return 2
    elif age > 20:
        return 5
    else:
        return 2              # for ages between 10 and 20 (not specified)


In [ ]:
def score_multisite(nb_etablissements):
    if nb_etablissements' >  1:
        return 5
    else:
        return 0

In [ ]:
def score_legal (forme_jur):
    if forme_jur in ['SA', 'SAS']
        return 5
    else:
        return 0


In [ ]:
score_sector
score_territory
score_age()
score_multisite("nbEtabSecondaire")
score_legal


In [ ]:
import numpy as np
import pandas as pd

# ------------------------------------------------------
# 1. SIGNAL WEIGHTS (from client spreadsheet)
# ------------------------------------------------------
signal_weights = {
    'B': 25,   # Construction (usine)
    'X': 15,   # Foncier / bâti
    'E': 20,   # Création (nouvelle usine)
    'F': 15,   # Croissance
    'N': 10,   # Recrutement
    'S': 7,    # Lancement
    'K1': 8,   # Investissements

    # Red flags
    'O': -15,  # Fermeture d’établissement
    'M': -20,  # Licenciements / chômage technique
    'I': -50   # Liquidation / RJ / LJ
}

# Score for each signal
df['signal_points'] = df['signal_code'].map(signal_weights).fillna(0)

# ------------------------------------------------------
# 2. AGE SCORING
# ------------------------------------------------------
df['age_points'] = np.select(
    [
        df['age'] <= 3,
        df['age'] <= 10,
        df['age'] > 20
    ],
    [
        0,   # 0–3 ans
        2,   # 3–10 ans
        5    # +20 ans
    ],
    default=2  # 10–20 ans
)

# ------------------------------------------------------
# 3. MULTISITE SCORING
# ------------------------------------------------------
df['multisite_points'] = np.where(df['nb_etablissements'] > 1, 5, 0)

# ------------------------------------------------------
# 4. LEGAL FORM SCORING
# ------------------------------------------------------
df['legal_points'] = np.where(df['forme_jur'].isin(['SA', 'SAS']), 5, 0)

# ------------------------------------------------------
# 5. SECTOR COEFFICIENT
# ------------------------------------------------------
sector_coeff = {
    'Industrie manufacturière': 1.0,
    'Agroalimentaire': 1.2,
    'Énergie / Chimie / Pharma': 1.3,
    'Logistique / Transport': 0.9,
    'ESN / Services / Conseil / Logiciel': 0.6
}
df['sector_coeff'] = df['sector'].map(sector_coeff).fillna(1.0)

# ------------------------------------------------------
# 6. TERRITORY COEFFICIENT
# ------------------------------------------------------
territory_coeff = {
    'Zone industrielle dynamique': 1.1,
    'Zone rurale neutre': 1.0,
    'Zone en décroissance': 0.8
}
df['territory_coeff'] = df['territory'].map(territory_coeff).fillna(1.0)


# 7. BASE SCORE (additive part)
# ------------------------------------------------------
df['base_score'] = (
    df['signal_points']
    + df['age_points']
    + df['multisite_points']
    + df['legal_points']
)


# 8. RAW SCORE (after multipliers)
# ------------------------------------------------------
df['raw_score'] = (
    df['base_score']
    * df['sector_coeff']
    * df['territory_coeff']
)


# 9. NORMALISED SCORE 0–100
# ------------------------------------------------------
min_raw = df['raw_score'].min()
max_raw = df['raw_score'].max()

df['score_100'] = (df['raw_score'] - min_raw) / (max_raw - min_raw) * 100
